# GRU: NER


In [ ]:
import collections
import json
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from torch.utils.data import DataLoader
from transformers import PreTrainedTokenizerFast, TrainingArguments

from core.args import DataArguments, ModelArguments
from core.datasets import (
    PAD_TOKEN,
    UNK_TOKEN,
    DataCollatorForTokenClassification,
    TextDataset,
    build_word_tokenizer,
    get_label_distribution,
    read_conll_file,
    simple_tokenizer,
    yield_tokens,
)
from core.model import UniversalRNNTokenClassifier
from core.predict import infer
from helpers.config import CONFIG_DIR, DATA_DIR, PROCESSED_DIR, parse_config

NER_DIR = DATA_DIR / "ukr-synth" / "data"
OUT_DIR = PROCESSED_DIR / "lecture" / "lstm_gru_ner"
assert NER_DIR.exists(), f"{NER_DIR} missing: download via hf CLI"
OUT_DIR.mkdir(parents=True, exist_ok=True)

model_args, data_args, training_args = parse_config(
    CONFIG_DIR / "ner_gru.yaml",
    (ModelArguments, DataArguments, TrainingArguments),
)

## Data


In [ ]:
train_data = read_conll_file(NER_DIR / "train.conllu.gz", nrows=data_args.train_rows)
valid_data = read_conll_file(NER_DIR / "dev.conllu.gz", nrows=data_args.valid_rows)

print("Number of train sentences:", len(train_data))
print("Number of valid sentences:", len(valid_data))
print("Sentence example:", train_data[1])

In [ ]:
columns = [
    "ID",
    "FORM",
    "LEMMA",
    "UPOS",
    "XPOS",
    "FEATS",
    "HEAD",
    "DEPREL",
    "DEPS",
    "MISC",
]

sentence = pd.DataFrame(train_data[1][2:], columns=columns)
sentence

In [ ]:
label_dist = get_label_distribution(train_data)

print("Unique labels:")
print(list(label_dist.keys()))

print("\nLabel distribution:")
for label, count in label_dist.items():
    print(f"{label}: {count}")

## Vocab


In [ ]:
tokenizer = build_word_tokenizer(
    yield_tokens(train_data + valid_data), specials=(UNK_TOKEN, PAD_TOKEN)
)

# Example usage:
tokens = simple_tokenizer(train_data[1])
token_indices = tokenizer.convert_tokens_to_ids(tokens)

print("Tokens:", tokens)
print("Token indices:", token_indices)

In [ ]:
len(tokenizer)

In [ ]:
tokenizer.convert_tokens_to_ids(["поки", "що", "покищо", "покищо#", "<unk>", "<pad>"])

In [ ]:
token_freqs = collections.Counter(
    token for tokens in yield_tokens(train_data) for token in tokens
)

# Get the 30 most common tokens
top30 = token_freqs.most_common(30)
top30

## Dataset


In [ ]:
# Here we want to figure out empirically max_length

train_n_tokens = pd.Series([len(simple_tokenizer(s)) for s in train_data])

plt.title("Train N Tokens Distribution")
train_n_tokens.hist(bins=100)
plt.show()

max_length = np.quantile(train_n_tokens, data_args.max_length_quantile)
print(f"{data_args.max_length_quantile} Quantile of N tokens: {max_length}")

In [ ]:
ds_train = TextDataset(train_data, tokenizer, list(label_dist.keys()), max_length)
ds_valid = TextDataset(valid_data, tokenizer, list(label_dist.keys()), max_length)

In [ ]:
pad_token_id = tokenizer.pad_token_id
collator = DataCollatorForTokenClassification(pad_token_id, label_pad_token_id=-100)

dataloader = DataLoader(
    ds_train, batch_size=training_args.per_device_train_batch_size, collate_fn=collator
)

In [ ]:
data_iter = iter(dataloader)
first_batch = next(data_iter)
first_batch

## Inference

Trained by `scripts/train_ner.py`.


In [ ]:
out_dir = Path(training_args.output_dir)

if not (out_dir / "eval_results.json").exists():
    raise FileNotFoundError("Model was not trained yet.")

best_model = UniversalRNNTokenClassifier.from_pretrained(out_dir).eval()

tokenizer = PreTrainedTokenizerFast.from_pretrained(out_dir)

print("Eval:", json.loads((out_dir / "eval_results.json").read_text()))

In [ ]:
infer(best_model, "Привіт, як у тебе справи?", tokenizer)

In [ ]:
text = """Палата відділення щелепнолицевої хірургії по вулиці Зоологічній,
в палаті на кроватях лежать Василь Гнатович і Енгельс Гасанович"""
infer(best_model, text, tokenizer)

In [ ]:
tokenizer.convert_tokens_to_ids(
    ["Зоологічній", "Василь", "Гнатович", "Енгельс", "Гасанович"]
)

In [ ]:
text = """Недалеко от Богуслава, коло Росі, в довгому покрученому яру розкинулось село Семигори.
Яр в'ється гадюкою між крутими горами, між зеленими терасами;
од яру на всі боки розбіглись, неначе гілки дерева, глибокі рукави й поховались десь далеко в густих лісах."""
infer(best_model, text, tokenizer)